In [11]:
# %% [markdown]
# # Lagged dual-encoder S2S downscaling — decomposed
#
# Cells run top-to-bottom once. After that, re-run only the cell you changed
# plus Cell 10 (the runner). State lives in notebook globals on purpose, so
# you can inspect anything at any point.
#
# | cell | owns | re-run when |
# |---|---|---|
# | 1 | config | every experiment |
# | 2 | utils | ~never |
# | 3 | climatology / anomalies | rarely |
# | 4 | prepare (archive → cache) | new predictors |
# | 5 | cache load, grids, dataset | perf tuning |
# | 6 | model | architecture experiments |
# | 7 | loss | loss experiments |
# | 8 | metrics | ~never |
# | 9 | train / predict | rarely |
# | 10 | CV runner | every experiment |
# | 11 | plots | ~never |

In [18]:
import pandas as pd
import xarray as xr
ds_big = xr.open_zarr("/Users/abhimanyu/Downloads/IFS_reforecast_download-main/s2s_new_vars_sorted.zarr")
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)

In [12]:



# %%
# ============ CELL 1: CONFIG ============
# The only cell you edit between experiments.

import gc, json, os, time
from contextlib import contextmanager
import numpy as np

DTYPE = np.float32
IMD_TARGET_VAR = "rain"

# --- data / windows ---
WINDOWS   = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
LAG_DAYS  = [0, 7, 14, 21, 28, 35]   # [0] = no lags (the control run)
LAG_TOL_DAYS = 1
MONTHS    = None                     # None = full year; (6,7,8,9) = JJAS
COARSE_PAD = 3.0
CLIM_WINDOW_DAYS = 7
TEST_YEARS_N = 3

BIG_VARS = ["top_net_thermal_radiation", "geopotential_height_200",
            "geopotential_height_500", "geopotential_height_850",
            "geopotential_height_1000"]

# --- model / training ---
BASE      = 24
DROP      = 0.2
BATCH     = 8
EPOCHS    = 60
LR        = 2e-4
WD        = 1e-3
PATIENCE  = 10
FOLDS     = 5
USE_LAG_MIXER = True    # False -> feed lag channels straight to the 3x3 conv.
                        # The 1x1 mixer is an ARCHITECTURAL change vs the
                        # no-lag baseline; set False to keep the comparison
                        # clean, True to keep params down.

# --- loss ---
LOSS_NAME = "mse"       # "mse" | "regime"
REGIME_W  = (0.2, 0.2, 0.6)   # light / moderate / heavy
W_BASE, W_REGIME, W_AGG = 1.0, 1.0, 0.3

# --- paths (tag them so runs never collide or resume each other) ---
TAG      = f"lag{len(LAG_DAYS)}_{LOSS_NAME}"
CACHE    = "../data/cache/unet_cache_lag"          # a DIRECTORY of .npy files
OUT_MAPS = f"../results/models/unet_{TAG}.nc"

print(f"TAG={TAG} | lags={LAG_DAYS} | months={MONTHS} | loss={LOSS_NAME}")



TAG=lag6_mse | lags=[0, 7, 14, 21, 28, 35] | months=None | loss=mse


In [13]:

# %%
# ============ CELL 2: UTILS ============
# Never changes.

@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


def normalize_step(ds, verbose=True):
    """zarr round-trips lose timedelta encoding; step comes back as bare int."""
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    """Verify rather than trust: a wrong valid_time silently pairs forecasts
    with the wrong day's rain."""
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


In [14]:


# %%
# ============ CELL 3: CLIMATOLOGY & ANOMALIES ============
# Leakage-critical: everything here takes `rows` = the TRAIN subset only.
#
# NOTE on lag channels: the climatology is computed PER CHANNEL, and channel
# block L always holds the field from exactly lag_L days before the target.
# So clim[doy, L] is already the climatology of (doy - lag_L). Indexing it
# with the target doy is correct, not a bug.

def _clim_grid(values, doys, window, rows=None, cell_chunk=20000, row_chunk=512):
    """(n, ...) -> (366, ...) DOY climatology.

    Chunked over cells AND rows so `values` can be a memmap and the train
    subset is never materialised. Not bit-identical to a single matmul
    (float32 addition reassociates over row chunks); differences ~1e-7.
    """
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    n_cells = v2.shape[1]
    if rows is None:
        rows = np.arange(len(values))
    elif np.asarray(rows).dtype == bool:
        rows = np.where(rows)[0]
    rows = np.asarray(rows)

    centers = np.arange(1, 367)
    d = np.abs(doys[rows][None, :].astype(np.int32) - centers[:, None])
    M = (np.minimum(d, 366 - d) <= window).astype(DTYPE)

    out = np.empty((366, n_cells), DTYPE)
    for a in range(0, n_cells, cell_chunk):
        b = min(a + cell_chunk, n_cells)
        counts = np.zeros((366, b - a), DTYPE)
        sums = np.zeros((366, b - a), DTYPE)
        for r0 in range(0, len(rows), row_chunk):
            r1 = min(r0 + row_chunk, len(rows))
            blk = np.asarray(v2[rows[r0:r1], a:b])
            fin = np.isfinite(blk)
            counts += M[:, r0:r1] @ fin.astype(DTYPE)
            sums += M[:, r0:r1] @ np.where(fin, blk, 0).astype(DTYPE)
            del blk, fin
        with np.errstate(invalid="ignore", divide="ignore"):
            out[:, a:b] = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
        del counts, sums
    return out.reshape((366,) + shp)


def anomalise_lagged(X, doy, tr, chunk=256):
    """X: (N, n_lag, V, h, w) -> (N, n_lag*V, h, w) anomalised + standardised.
    Lags are folded into the channel axis. In-place chunked: no full-size
    clim[doy-1] broadcast, no triple copy."""
    N, nl, V, h, w = X.shape
    flat = X.reshape(N, nl * V, h, w)
    clim = _clim_grid(flat, doy, CLIM_WINDOW_DAYS, rows=tr)

    out = np.empty((N, nl * V, h, w), DTYPE)
    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        out[a:b] = flat[a:b] - clim[doy[a:b] - 1]
    del clim

    idx = np.where(tr)[0]
    s1 = np.zeros(nl * V, np.float64); s2 = np.zeros(nl * V, np.float64); cnt = 0
    for a in range(0, len(idx), chunk):
        blk = out[idx[a:a + chunk]]
        s1 += np.nansum(blk, axis=(0, 2, 3))
        s2 += np.nansum(blk.astype(np.float64) ** 2, axis=(0, 2, 3))
        cnt += blk.shape[0] * blk.shape[2] * blk.shape[3]
    m = (s1 / cnt).astype(DTYPE)[None, :, None, None]
    sd = np.sqrt(np.maximum(s2 / cnt - (s1 / cnt) ** 2, 0)).astype(DTYPE)
    sd = np.where(sd < 1e-8, 1.0, sd)[None, :, None, None]

    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        np.subtract(out[a:b], m, out=out[a:b])
        np.divide(out[a:b], sd, out=out[a:b])
        np.nan_to_num(out[a:b], copy=False)
    return out


def anomalise_target(y, doy, tr, chunk=256):
    """Keeps NaN — the mask depends on it."""
    clim = _clim_grid(y, doy, CLIM_WINDOW_DAYS, rows=tr)
    out = np.empty(y.shape, DTYPE)
    for a in range(0, len(y), chunk):
        b = min(a + chunk, len(y))
        out[a:b] = y[a:b] - clim[doy[a:b] - 1]
    del clim
    return out


In [15]:
# %%
# ============ CELL 4: PREPARE (archive -> cache) ============
# Run once per predictor set. Writes a DIRECTORY of .npy files, not an .npz,
# because mmap_mode is silently ignored for npz members.

def _subset_box(ds, imd, pad):
    ds = ensure_valid_time(ds)
    for c in ("lat", "lon"):
        if ds[c].values[0] > ds[c].values[-1]:
            ds = ds.sortby(c)
    if pad is not None:
        la0, la1 = float(imd.lat.min()), float(imd.lat.max())
        lo0, lo1 = float(imd.lon.min()), float(imd.lon.max())
        ds = ds.sel(lat=slice(la0 - pad, la1 + pad), lon=slice(lo0 - pad, lo1 + pad))
    return ds


def _window_stack(sub, feature_vars, leads, lo, hi):
    from dask.diagnostics import ProgressBar
    sel = np.where((leads >= lo) & (leads <= hi))[0]
    if len(sel) == 0:
        raise ValueError(f"no leads in [{lo},{hi}]")
    with ProgressBar():
        Xw = sub.isel(step=sel).mean(dim="step").compute()
    return np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE), sel


def _build_lag_index(all_inits, keep_mask, lag_days, tol):
    """idx[i, L] = row of the init nearest to keep_init[i] - lag_days[L].
    ok[i] False if ANY lag missing -> row dropped, never padded (zero in
    anomaly space means 'exactly climatological', a false claim)."""
    day = np.timedelta64(1, "D")
    kept = np.where(keep_mask)[0]
    idx = np.zeros((len(kept), len(lag_days)), np.int64)
    ok = np.ones(len(kept), bool)
    for L, lag in enumerate(lag_days):
        target = all_inits[kept] - lag * day
        pos = np.clip(np.searchsorted(all_inits, target), 1, len(all_inits) - 1)
        lo_ = all_inits[pos - 1]
        hi_ = all_inits[np.minimum(pos, len(all_inits) - 1)]
        pick_hi = np.abs(hi_ - target) < np.abs(target - lo_)
        chosen = np.where(pick_hi, np.minimum(pos, len(all_inits) - 1), pos - 1)
        ok &= np.abs(all_inits[chosen] - target) / day <= tol
        idx[:, L] = chosen
    return kept, idx, ok


def prepare(ecmwf_ds, big_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar
    from numpy.lib.format import open_memmap

    imd = imd_ds[IMD_TARGET_VAR].assign_coords(
        time=imd_ds[IMD_TARGET_VAR]["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    with stage("Subset both boxes (full year — lags need pre-season history)"):
        subA = _subset_box(ecmwf_ds, imd, COARSE_PAD)
        subB = _subset_box(big_ds, imd, pad=None)
        varsA = list(subA.data_vars)
        missing = [v for v in BIG_VARS if v not in subB.data_vars]
        if missing:
            raise KeyError(f"BIG_VARS missing: {missing}")
        varsB = BIG_VARS
        clatA, clonA = subA["lat"].values, subA["lon"].values
        clatB, clonB = subB["lat"].values, subB["lon"].values
        leadsA = (subA["step"].values / np.timedelta64(1, "D")).astype(int)
        leadsB = (subB["step"].values / np.timedelta64(1, "D")).astype(int)
        initA, initB = subA["time"].values, subB["time"].values
        print(f"    A: {len(varsA)} vars {len(clatA)}x{len(clonA)} | "
              f"B: {len(varsB)} vars {len(clatB)}x{len(clonB)}")

    # guards
    if initA.shape != initB.shape or not (initA == initB).all():
        raise ValueError("A and B have different init dates")
    for cla, clo, nm in [(clatA, clonA, "A"), (clatB, clonB, "B")]:
        if not (cla.min() <= flat_lat.min() and cla.max() >= flat_lat.max()
                and clo.min() <= flat_lon.min() and clo.max() >= flat_lon.max()):
            raise ValueError(f"IMD grid not inside box {nm} — grid_sample would clamp")

    with stage("Lag index"):
        month = initA.astype("datetime64[M]").astype(int) % 12 + 1
        keep = np.isin(month, MONTHS) if MONTHS else np.ones(len(initA), bool)
        kept, lag_idx, ok = _build_lag_index(initA, keep, LAG_DAYS, LAG_TOL_DAYS)
        kept, lag_idx = kept[ok], lag_idx[ok]
        print(f"    {len(kept)} usable inits ({int((~ok).sum())} dropped for "
              f"incomplete {max(LAG_DAYS)}d history)")
        if len(kept) == 0:
            raise ValueError("no inits have complete history")

    os.makedirs(cache_path, exist_ok=True)
    n_kept, nl, nw = len(kept), len(LAG_DAYS), len(WINDOWS)
    XA_mm = XB_mm = None
    y_l, doy_l, wid_l = [], [], []

    for wi, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi}) x {nl} lags"):
            fullA, sel = _window_stack(subA, varsA, leadsA, lo, hi)
            fullB, _ = _window_stack(subB, varsB, leadsB, lo, hi)
            if XA_mm is None:
                XA_mm = open_memmap(f"{cache_path}/XA.npy", mode="w+", dtype=DTYPE,
                                    shape=(n_kept * nw, nl) + fullA.shape[1:])
                XB_mm = open_memmap(f"{cache_path}/XB.npy", mode="w+", dtype=DTYPE,
                                    shape=(n_kept * nw, nl) + fullB.shape[1:])
            XA_mm[wi * n_kept:(wi + 1) * n_kept] = fullA[lag_idx]
            XB_mm[wi * n_kept:(wi + 1) * n_kept] = fullB[lag_idx]
            del fullA, fullB; gc.collect()

            vt = subA["valid_time"].isel(time=kept, step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            y_l.append(np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1))
            del y_all
            centre = initA[kept] + np.timedelta64((lo + hi) // 2, "D")
            doy_l.append(xr.DataArray(centre, dims="t").dt.dayofyear.values)
            wid_l.append(np.full(len(kept), wi, np.int64))

    XA_mm.flush(); XB_mm.flush()
    y = np.concatenate(y_l); doy = np.concatenate(doy_l); wid = np.concatenate(wid_l)
    year = np.tile(initA[kept].astype("datetime64[Y]").astype(int) + 1970, nw)

    with stage("Mask + test holdout + cache"):
        mask = np.isfinite(y).all(axis=0)      # strict: valid on every sample
        uy = np.unique(year)
        is_test = np.isin(year, list(uy[-TEST_YEARS_N:]))
        print(f"    strict mask {int(mask.sum())} cells | test {sorted(uy[-TEST_YEARS_N:])}")
        np.save(f"{cache_path}/y.npy", y)
        np.savez(f"{cache_path}/meta.npz", doy=doy, wid=wid, year=year,
                 mask=mask, is_test=is_test, clatA=clatA, clonA=clonA,
                 clatB=clatB, clonB=clonB, flat_lat=flat_lat, flat_lon=flat_lon,
                 varsA=np.array(varsA), varsB=np.array(varsB),
                 lag_days=np.array(LAG_DAYS),
                 window_names=np.array([w[0] for w in WINDOWS]))
        tot = sum(os.path.getsize(f"{cache_path}/{f}") for f in os.listdir(cache_path))
        print(f"    {tot/1e9:.2f} GB -> {cache_path}/")


# RUN ONCE (uncomment):
prepare(ds_ecmv, ds_big, ds_imd, CACHE)


NameError: name 'ds_ecmv' is not defined

In [ ]:
# ============ CELL 5: LOAD CACHE + GRIDS + DATASET ============
# Re-run after prepare, or when tuning dataloader performance.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DEV = ("cuda" if torch.cuda.is_available()
       else "mps" if torch.backends.mps.is_available() else "cpu")

# XA/XB stay on disk: they are only read SEQUENTIALLY (anomalise walks them in
# row chunks), so paging is cheap. The random-access arrays the DataLoader hits
# are XAa/XBa, which cell 10 allocates in RAM.
XA = np.load(f"{CACHE}/XA.npy", mmap_mode="r")
XB = np.load(f"{CACHE}/XB.npy", mmap_mode="r")
y  = np.load(f"{CACHE}/y.npy")
_m = np.load(f"{CACHE}/meta.npz")

doy, wid = np.asarray(_m["doy"]), np.asarray(_m["wid"])
year, mask, is_test = np.asarray(_m["year"]), np.asarray(_m["mask"]), np.asarray(_m["is_test"])
flat_lat, flat_lon = np.asarray(_m["flat_lat"]), np.asarray(_m["flat_lon"])
clatA, clonA, clatB, clonB = _m["clatA"], _m["clonA"], _m["clatB"], _m["clonB"]
window_names = [str(w) for w in _m["window_names"]]

H, W = len(flat_lat), len(flat_lon)
N_WIN = len(window_names)
N_LAG = XA.shape[1]
C_A, C_B = XA.shape[2] * N_LAG, XB.shape[2] * N_LAG   # lags folded to channels


def _make_samp(clat, clon):
    """Coordinate-aware bilinear grid: maps the fine IMD grid into a source
    box's normalised [-1,1] frame. Asserting |g|<=1 catches an extent bug
    that would otherwise silently edge-clamp."""
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    s = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(s).max() <= 1.0, "fine grid outside source box"
    return torch.tensor(s).to(DEV)


SAMP_A, SAMP_B = _make_samp(clatA, clonA), _make_samp(clatB, clonB)

_lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
_lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
STATIC = torch.tensor(np.stack([
    mask.astype(DTYPE),
    np.broadcast_to(_lat2, (H, W)).astype(DTYPE),
    np.broadcast_to(_lon2, (H, W)).astype(DTYPE)])[None]).to(DEV)

# strict mask means isfinite(y[i]) & mask == mask for every i, so this is
# constant — recomputing it per __getitem__ was pure waste
FIN_STATIC = mask.astype(DTYPE)


class LagDS(Dataset):
    """from_numpy shares memory (torch.tensor copies). yb built per item so a
    full-size yt array never exists."""
    def __init__(self, XAa, XBa, ya, idx):
        self.XAa, self.XBa, self.ya, self.idx = XAa, XBa, ya, idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return (torch.from_numpy(np.ascontiguousarray(self.XAa[j])),
                torch.from_numpy(np.ascontiguousarray(self.XBa[j])),
                int(wid[j]),
                torch.from_numpy(np.nan_to_num(self.ya[j])),
                torch.from_numpy(FIN_STATIC))


print(f"device {DEV} | {len(XA)} samples | encA {C_A}ch encB {C_B}ch | "
      f"grid {H}x{W} | {int(mask.sum())} valid cells")



device mps | 11106 samples | encA 126ch encB 30ch | grid 129x135 | 4964 valid cells


In [6]:
# %%
# ============ CELL 6: MODEL ============
# Edit for architecture experiments.

def gn(c):
    for g in (8, 4, 2, 1):
        if c % g == 0:
            return nn.GroupNorm(g, c)


class Block(nn.Module):
    def __init__(self, ci, co, drop=0.0):
        super().__init__()
        L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
             nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
        if drop > 0:
            L.append(nn.Dropout2d(drop))
        self.f = nn.Sequential(*L)

    def forward(self, x):
        return self.f(x)


class LaggedDualUNet(nn.Module):
    """Two coarse encoders on DIFFERENT boxes, each grid_sampled to the shared
    fine grid with its own sampling grid, concatenated there, then one decoder.

    Merging at the fine grid (not the bottleneck) is what lets the two extents
    coexist: each encoder reasons spatially in its own frame first.

    Lags enter as channels (n_lag x V) so the first conv can form differences
    across lags — that is the tendency signal. USE_LAG_MIXER=True inserts a 1x1
    to compress them first (fewer params); False feeds them straight to the 3x3
    (matches the no-lag baseline's structure, so the lag ablation is clean).
    """
    def __init__(self, cA, cB, n_win, base=24, drop=0.2, emb=4, mixer=True):
        super().__init__()
        self.emb = nn.Embedding(n_win, emb)
        self.mixer = mixer
        if mixer:
            self.mixA = nn.Conv2d(cA, base * 2, 1)
            self.mixB = nn.Conv2d(cB, base * 2, 1)
            inA, inB = base * 2 + emb, base * 2
        else:
            inA, inB = cA + emb, cB
        self.encA1 = Block(inA, base * 2); self.encA2 = Block(base * 2, base * 2)
        self.encB1 = Block(inB, base * 2); self.encB2 = Block(base * 2, base * 2)
        self.inp = Block(base * 4 + 3, base)
        self.d1 = Block(base, base * 2, drop)
        self.d2 = Block(base * 2, base * 4, drop)
        self.bott = Block(base * 4, base * 4, drop)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.du2 = Block(base * 4, base * 2, drop)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.du1 = Block(base * 2, base)
        self.head = nn.Conv2d(base, 1, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, xa, xb, w, sampA, sampB, static):
        b = xa.shape[0]
        if self.mixer:
            xa, xb = self.mixA(xa), self.mixB(xb)
        e = self.emb(w)[:, :, None, None].expand(-1, -1, xa.shape[2], xa.shape[3])
        ca = self.encA2(self.encA1(torch.cat([xa, e], 1)))
        cb = self.encB2(self.encB1(xb))
        fa = F.grid_sample(ca, sampA.expand(b, -1, -1, -1),
                           mode="bilinear", align_corners=True)
        fb = F.grid_sample(cb, sampB.expand(b, -1, -1, -1),
                           mode="bilinear", align_corners=True)
        f = torch.cat([fa, fb, static.expand(b, -1, -1, -1)], 1)
        H0, W0 = f.shape[-2:]
        f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
        e0 = self.inp(f)
        e1 = self.d1(self.pool(e0))
        e2 = self.d2(self.pool(e1))
        u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], 1))
        u = self.du1(torch.cat([self.u1(u), e0], 1))
        return self.head(u)[:, :, :H0, :W0].squeeze(1)


def new_model():
    m = LaggedDualUNet(C_A, C_B, N_WIN, base=BASE, drop=DROP,
                       mixer=USE_LAG_MIXER).to(DEV)
    return m


print(f"params: {sum(p.numel() for p in new_model().parameters())/1e6:.2f}M "
      f"(mixer={USE_LAG_MIXER})")

params: 0.63M (mixer=True)


In [7]:
# %%
# ============ CELL 7: LOSS ============
# Edit for loss experiments. Every loss returns (scalar, stats_dict).

def masked_mse(pred, target, m):
    """Fill-then-mask-then-normalise-BY-MASK. Never by numel, or the loss
    depends on how much ocean is in the domain."""
    se = (pred - target) ** 2 * m
    return se.sum() / m.sum().clamp(min=1.0), {}


def compute_pixel_thresholds(ya, tr, mask, pct=(10.0, 90.0), chunk=20000):
    """Per-pixel R10/R90 of the target ANOMALY, TRAIN samples only.

    Per-pixel, not global: 'heavy' in the Thar and 'heavy' in the Ghats are
    different numbers, and a global threshold would park the whole arid
    northwest permanently in the 'light' bucket.
    """
    flat = ya[tr].reshape(int(np.sum(tr)), -1)
    n_cells = flat.shape[1]
    r_lo = np.full(n_cells, np.nan, DTYPE)
    r_hi = np.full(n_cells, np.nan, DTYPE)
    mf = mask.reshape(-1)
    for a in range(0, n_cells, chunk):
        b = min(a + chunk, n_cells)
        valid = mf[a:b]
        if not valid.any():
            continue
        sub = flat[:, a:b][:, valid]
        with np.errstate(invalid="ignore"):
            r_lo[np.where(valid)[0] + a] = np.nanpercentile(sub, pct[0], axis=0)
            r_hi[np.where(valid)[0] + a] = np.nanpercentile(sub, pct[1], axis=0)
    return r_lo.reshape(mask.shape), r_hi.reshape(mask.shape)


class RegimeLoss(nn.Module):
    """base MSE + regime-partitioned MSE (per-pixel percentiles) + a
    spatial-aggregate term on the domain-mean anomaly.

    The aggregate term exists because per-cell MSE and domain-total error are
    different failures: errors that all lean one way cancel badly in the total.
    Your two reported metrics (per-cell ACC, India-mean corr) measure exactly
    these two things; plain MSE only optimises the first.
    """
    def __init__(self, r_lo, r_hi):
        super().__init__()
        self.register_buffer("r_lo", torch.as_tensor(np.nan_to_num(r_lo, nan=-1e9)))
        self.register_buffer("r_hi", torch.as_tensor(np.nan_to_num(r_hi, nan=+1e9)))
        self.wl, self.wm, self.wh = REGIME_W

    def forward(self, pred, target, m):
        se = (pred - target) ** 2
        base = (se * m).sum() / m.sum().clamp(min=1.0)

        lo, hi = self.r_lo.unsqueeze(0), self.r_hi.unsqueeze(0)
        m_low = m * (target < lo).float()
        m_high = m * (target > hi).float()
        m_mid = m * ((target >= lo) & (target <= hi)).float()
        # each regime normalised by ITS OWN count, so a rare regime is not
        # automatically negligible
        l_low = (se * m_low).sum() / m_low.sum().clamp(min=1.0)
        l_mid = (se * m_mid).sum() / m_mid.sum().clamp(min=1.0)
        l_high = (se * m_high).sum() / m_high.sum().clamp(min=1.0)
        regime = self.wl * l_low + self.wm * l_mid + self.wh * l_high

        wsum = m.sum((1, 2)).clamp(min=1.0)
        agg = (((pred * m).sum((1, 2)) / wsum
                - (target * m).sum((1, 2)) / wsum) ** 2).mean()

        total = W_BASE * base + W_REGIME * regime + W_AGG * agg
        return total, {"base": float(base.detach()), "high": float(l_high.detach()),
                       "agg": float(agg.detach())}


def make_criterion(ya, tr):
    """Built per fold — thresholds must come from that fold's train years."""
    if LOSS_NAME == "mse":
        return masked_mse
    if LOSS_NAME == "regime":
        r_lo, r_hi = compute_pixel_thresholds(ya, tr, mask)
        return RegimeLoss(r_lo, r_hi).to(DEV)
    raise ValueError(LOSS_NAME)


In [8]:
# %%
# ============ CELL 8: METRICS ============
# Never changes.

def skill_acc(p, t, fin):
    """Per-cell skill vs zero-anomaly climatology, and per-cell ACC.
    rc>1e-6 guard: a degenerate near-constant cell would give skill = -inf
    and poison the whole nanmean."""
    se_m = np.where(fin, (t - p) ** 2, np.nan)
    se_c = np.where(fin, t ** 2, np.nan)
    with np.errstate(invalid="ignore"):
        rm = np.sqrt(np.nanmean(se_m, axis=0))
        rc = np.sqrt(np.nanmean(se_c, axis=0))
        ok = rc > 1e-6
        skill = np.where(ok, 1 - rm / np.where(ok, rc, 1), np.nan)
        tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
        pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
        num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
        den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                      * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
        acc = np.where(den > 0, num / den, np.nan)
    return skill, acc


def std_ratio(p, t, fin):
    """Diagnostic only — reported, never optimised. <~0.8 means the model is
    hedging amplitude (MSE regressing toward the mean)."""
    pv, tv = p[fin], t[fin]
    return float(pv.std() / tv.std()) if tv.std() > 0 else np.nan



In [9]:
# %%
# ============ CELL 9: TRAIN / PREDICT ============
# Rarely changes.

def train_one(tr_idx, va_idx, XAa, XBa, ya, crit, max_epochs=EPOCHS, verbose=False):
    # num_workers=0: LagDS closes over notebook globals, and spawn would copy
    # XAa/XBa (~GBs) into every worker — worse than the problem it solves.
    tr_dl = DataLoader(LagDS(XAa, XBa, ya, tr_idx), batch_size=BATCH, shuffle=True)
    va_dl = DataLoader(LagDS(XAa, XBa, ya, va_idx), batch_size=BATCH, shuffle=False)

    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    best, best_state, wait = np.inf, None, 0

    for ep in range(max_epochs):
        model.train()
        for xa, xb, w, yb, mb in tr_dl:
            xa, xb, w, yb, mb = [t.to(DEV) for t in (xa, xb, w, yb, mb)]
            loss, _ = crit(model(xa, xb, w, SAMP_A, SAMP_B, STATIC), yb, mb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        vl, n, st_last = 0.0, 0, {}
        with torch.no_grad():
            for xa, xb, w, yb, mb in va_dl:
                xa, xb, w, yb, mb = [t.to(DEV) for t in (xa, xb, w, yb, mb)]
                l, st_last = crit(model(xa, xb, w, SAMP_A, SAMP_B, STATIC), yb, mb)
                vl += float(l) * len(xa); n += len(xa)
        vl /= n

        if vl < best - 1e-6:
            best, wait = vl, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose:
            extra = " ".join(f"{k} {v:.3f}" for k, v in st_last.items())
            print(f"      ep {ep:>2} vloss {vl:.4f} {extra}", flush=True)
        if wait >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best, ep + 1


def predict(model, idx, XAa, XBa, ya):
    dl = DataLoader(LagDS(XAa, XBa, ya, idx), batch_size=BATCH, shuffle=False)
    out = []
    model.eval()
    with torch.no_grad():
        for xa, xb, w, _, _ in dl:
            out.append(model(xa.to(DEV), xb.to(DEV), w.to(DEV),
                             SAMP_A, SAMP_B, STATIC).cpu().numpy())
    return np.concatenate(out)

In [10]:
# %%
# ============ CELL 10: CV RUNNER ============
# The cell you re-run for every experiment. Resumes from *_folds.json.

def run_fold(tr_i, va_i):
    """Anomalise on THIS fold's train years only, train, score."""
    XAa = anomalise_lagged(XA, doy, tr_i)
    XBa = anomalise_lagged(XB, doy, tr_i)
    ya = anomalise_target(y, doy, tr_i)
    crit = make_criterion(ya, tr_i)
    model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0], XAa, XBa, ya, crit)
    p = predict(model, np.where(va_i)[0], XAa, XBa, ya)
    t = ya[va_i]
    fin = np.isfinite(np.asarray(y)[va_i]) & mask[None]
    sk, ac = skill_acc(p, t, fin)
    res = dict(skill=float(np.nanmean(sk[mask])), acc=float(np.nanmean(ac[mask])),
               std_ratio=std_ratio(p, t, fin), epochs=int(eps), vloss=float(vloss))
    del XAa, XBa, ya, model, p, t; gc.collect()
    return res


resume_path = OUT_MAPS.replace(".nc", "_folds.json")
done = {}
if os.path.exists(resume_path):
    done = {int(k): v for k, v in json.load(open(resume_path)).items()}
    print(f"resuming: folds {sorted(done)} cached")

nontest_years = sorted(set(year[~is_test].tolist()))
blocks = np.array_split(nontest_years, FOLDS)

with stage(f"[{TAG}] CV: {FOLDS} folds over {len(nontest_years)} years"):
    for fi, vy_arr in enumerate(blocks):
        if fi in done:
            r = done[fi]
            print(f"  fold {fi} (cached): skill {r['skill']:+.3f} | ACC {r['acc']:.3f}")
            continue
        vy = set(vy_arr.tolist())
        va_i = np.isin(year, list(vy)) & ~is_test
        tr_i = ~np.isin(year, list(vy)) & ~is_test
        r = run_fold(tr_i, va_i)
        r["val_years"] = sorted(vy)
        done[fi] = r
        json.dump({str(k): v for k, v in done.items()}, open(resume_path, "w"), indent=2)
        print(f"  fold {fi} val {sorted(vy)}: skill {r['skill']:+.3f} | "
              f"ACC {r['acc']:.3f} | std {r['std_ratio']:.3f} | {r['epochs']} ep  [saved]",
              flush=True)

    ks = [i for i in range(FOLDS) if i in done]
    fs = [done[i]["skill"] for i in ks]; fa = [done[i]["acc"] for i in ks]
    print(f"\n  [{TAG}] CV skill {np.mean(fs):+.4f} +/- {np.std(fs):.4f} | "
          f"CV ACC {np.mean(fa):.4f} +/- {np.std(fa):.4f}")

[ ] [lag6_mse] CV: 5 folds over 17 years ...


KeyboardInterrupt: 

In [ ]:
# %%
# ============ CELL 10b: FINAL MODEL -> TEST ============

with stage("Final model on all non-test years -> test"):
    es_years = set(nontest_years[-2:])          # early-stopping slice
    va_i = np.isin(year, list(es_years)) & ~is_test
    fit_i = (~is_test) & ~va_i

    XAa = anomalise_lagged(XA, doy, fit_i)
    XBa = anomalise_lagged(XB, doy, fit_i)
    ya = anomalise_target(y, doy, fit_i)
    crit = make_criterion(ya, fit_i)
    model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0], XAa, XBa, ya, crit)

    te_i = np.where(is_test)[0]
    p = predict(model, te_i, XAa, XBa, ya)
    t = ya[is_test]
    fin = np.isfinite(np.asarray(y)[is_test]) & mask[None]

    import xarray as xr
    wid_te = wid[is_test]
    dvars, summary = {}, {}
    print(f"  trained {eps} ep")
    for w in range(N_WIN):
        sm = wid_te == w
        if sm.sum() == 0:
            continue
        sk, ac = skill_acc(p[sm], t[sm], fin[sm])
        wn = window_names[w]
        dvars[f"{wn}_skill"] = (("lat", "lon"), sk)
        dvars[f"{wn}_acc"] = (("lat", "lon"), ac)
        summary[wn] = dict(skill=float(np.nanmean(sk[mask])),
                           acc=float(np.nanmean(ac[mask])),
                           std_ratio=std_ratio(p[sm], t[sm], fin[sm]))
        print(f"  test {wn:>8}: skill {summary[wn]['skill']:+.4f} | "
              f"ACC {summary[wn]['acc']:.4f} | std {summary[wn]['std_ratio']:.3f} | "
              f"{100*np.nanmean(sk[mask]>0):.0f}% cells+")

    xr.Dataset(dvars, coords={"lat": flat_lat, "lon": flat_lon}).to_netcdf(OUT_MAPS)
    np.savez_compressed(OUT_MAPS.replace(".nc", "_pred.npz"),
                        pred=p, obs=t, wid=wid_te)
    torch.save({"state": model.state_dict(), "tag": TAG},
               OUT_MAPS.replace(".nc", ".pt"))

    # running comparison across every experiment you've run
    tbl = "../results/models/experiments.json"
    allr = json.load(open(tbl)) if os.path.exists(tbl) else {}
    allr[TAG] = dict(cv_skill=float(np.mean(fs)), cv_acc=float(np.mean(fa)),
                     cv_acc_std=float(np.std(fa)), test=summary,
                     lags=LAG_DAYS, loss=LOSS_NAME, months=str(MONTHS),
                     mixer=USE_LAG_MIXER, base=BASE)
    json.dump(allr, open(tbl, "w"), indent=2)
    print(f"  -> {OUT_MAPS} (+ _pred.npz, .pt); appended to {tbl}")

print(f"\n{'experiment':>22} {'CV ACC':>9} {'w3_4 ACC':>9} {'w5_6 ACC':>9}")
for k, v in sorted(json.load(open("../results/models/experiments.json")).items()):
    w34 = v["test"].get("week3_4", {}); w56 = v["test"].get("week5_6", {})
    print(f"{k:>22} {v['cv_acc']:>9.4f} {w34.get('acc', float('nan')):>9.4f} "
          f"{w56.get('acc', float('nan')):>9.4f}")

In [ ]:


# %%
# ============ CELL 11: PLOTS ============
# Reads the saved _pred.npz — no model rebuild, no inference.

def plot_all(pred_npz=None, tag=None):
    import matplotlib.pyplot as plt
    import xarray as xr
    pred_npz = pred_npz or OUT_MAPS.replace(".nc", "_pred.npz")
    tag = tag or TAG

    pr = np.load(pred_npz)
    unet_a, obs_a, wid_te = pr["pred"], pr["obs"], pr["wid"]

    # raw ECMWF tp -> fine grid -> anomaly (its own climatology; units cancel)
    varsA = [str(v) for v in _m["varsA"]]
    tp_i = varsA.index("total_precipitation")
    fit_i = ~is_test
    tp_c = np.asarray(XA[:, 0, tp_i])          # lag 0
    tp_f = xr.DataArray(tp_c, dims=("s", "lat", "lon"),
                        coords={"lat": clatA, "lon": clonA}
                        ).interp(lat=flat_lat, lon=flat_lon).values.astype(DTYPE)
    ecm_a = (tp_f - _clim_grid(tp_f, doy, CLIM_WINDOW_DAYS, rows=fit_i)[doy - 1])[is_test]

    m3 = mask[None]
    im = lambda a: np.nansum(np.where(m3, a, np.nan), axis=(1, 2)) / m3.sum()
    io, ie, iu = im(obs_a), im(ecm_a), im(unet_a)

    fig, axes = plt.subplots(N_WIN, 1, figsize=(13, 3.2 * N_WIN))
    for w, ax in enumerate(np.atleast_1d(axes)):
        s = wid_te == w
        tt = np.arange(s.sum())
        ax.plot(tt, io[s], "-", color="k", lw=2, label="observed IMD")
        ax.plot(tt, ie[s], "-", color="#888", lw=1.5, label="raw ECMWF")
        ax.plot(tt, iu[s], "-", color="#E45756", lw=1.5, label="UNet")
        ax.axhline(0, color="gray", lw=.5, ls=":")
        ru = np.corrcoef(iu[s], io[s])[0, 1]; re = np.corrcoef(ie[s], io[s])[0, 1]
        ax.set_title(f"{window_names[w]}  (India-mean; UNet {ru:.2f}, ECMWF {re:.2f})")
        ax.set_ylabel("anomaly (mm/day)")
        if w == 0:
            ax.legend(ncol=3, fontsize=9)
    fig.tight_layout(); fig.savefig(f"../results/figures/threeline_{tag}.png", dpi=140); plt.show()

    # per-cell correlation maps
    def cmap_(p_, t_, f_):
        with np.errstate(invalid="ignore", divide="ignore"):
            pm = np.nanmean(np.where(f_, p_, np.nan), 0)
            tm = np.nanmean(np.where(f_, t_, np.nan), 0)
            num = np.nansum(np.where(f_, (p_ - pm) * (t_ - tm), np.nan), 0)
            den = np.sqrt(np.nansum(np.where(f_, (p_ - pm) ** 2, np.nan), 0)
                          * np.nansum(np.where(f_, (t_ - tm) ** 2, np.nan), 0))
            return np.where(den > 0, num / den, np.nan)

    fin_all = np.isfinite(np.asarray(y)[is_test]) & m3
    maps = {}
    fig, axes = plt.subplots(N_WIN, 3, figsize=(15, 4.4 * N_WIN))
    ext = [flat_lon[0], flat_lon[-1], flat_lat[0], flat_lat[-1]]
    for w in range(N_WIN):
        s = wid_te == w
        ce = cmap_(ecm_a[s], obs_a[s], fin_all[s]); ce[~mask] = np.nan
        cu = cmap_(unet_a[s], obs_a[s], fin_all[s]); cu[~mask] = np.nan
        wn = window_names[w]
        maps[f"{wn}_corr_ecmwf"] = ce; maps[f"{wn}_corr_unet"] = cu
        maps[f"{wn}_corr_diff"] = cu - ce
        for c, (arr, ttl, cm, vr) in enumerate([
                (ce, "ECMWF vs IMD", "RdBu_r", .6), (cu, "UNet vs IMD", "RdBu_r", .6),
                (cu - ce, "UNet - ECMWF", "PuOr_r", .4)]):
            ax = np.atleast_2d(axes)[w, c]
            i_ = ax.imshow(arr, cmap=cm, origin="lower", extent=ext,
                           aspect="auto", vmin=-vr, vmax=vr)
            fig.colorbar(i_, ax=ax, fraction=.046)
            ax.set_title(f"{wn} {ttl}\n(mean {np.nanmean(arr[mask]):+.3f})", fontsize=9)
    fig.tight_layout(); fig.savefig(f"../results/figures/corr_maps_{tag}.png", dpi=130); plt.show()

    xr.Dataset({k: (("lat", "lon"), v) for k, v in maps.items()},
               coords={"lat": flat_lat, "lon": flat_lon}).to_netcdf(f"../results/models/corr_maps_{tag}.nc")
    np.savez_compressed(f"../results/models/corr_maps_{tag}.npz", lat=flat_lat, lon=flat_lon,
                        mask=mask, **maps)
    print(f"-> threeline_{tag}.png, corr_maps_{tag}.png/.nc/.npz")


# plot_all()